# Weighted Hybrid Recommendation System

This notebook implements a Hybrid Recommendation: CF+content models. It combines our previous two distinct approaches to improve our predictions for user preferences:
1. **Collaborative Filtering (CF):** Uses Singular Value Decomposition (SVD) to capture those latent user-item interaction patterns.
2. **Content-Based Filtering (CBF):** Builds the user and item profiles using restaurant metadata (categories, attributes) and text embeddings from user reviews.

The final recommendations are generated by weighting and then combining the scaled (min max) prediction scores from both models.

In [3]:
import pandas as pd
import numpy as np
import ast
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize, minmax_scale
from sentence_transformers import SentenceTransformer
from shared import load_data, evaluate_model

train_df, test_df, restaurants_df = load_data()

## Collaborative Filtering Setup (SVD)

Similar to our pure SVD approach, we construct the user-item rating matrix, impute missing values using the same cascaded mean strategy as before (user mean $\rightarrow$ item mean $\rightarrow$ global mean), and apply SVD to generate the matrix of predicted CF scores.

In [4]:
def prepare_svd_cf(train_df, restaurants_df, k=2):
    """
    Constructs the user-item interaction matrix, imputes missing values, and applies
    SVD to generate predictions.
    Inputs:
        train_df (pd.dataframe): Training data containing 'user_id', 'business_id', and 'stars'.
        restaurants_df (pd.dataframe) : Df of the unique restaurants.
        k (int): Number of latent factors for SVD. We set it to default to 2.
    Return vals:
        Tuple that contains:
            - predicted_ratings_cf (np.ndarray): The SVD-reconstructed prediction matrix.
            - R (np.ndarray): The original interaction matrix (used later for masking).
            - user_to_idx (dict): Mapping from user_id to row index.
            - idx_to_user (dict): Mapping from row index to user_id.
            - item_to_idx (dict): Mapping from business_id to column index.
            - idx_to_item (dict): Mapping from column index to business_id.
            - num_items (int): Total number of unique items.
    """
    user_ids = train_df['user_id'].unique()
    user_to_idx = {user: idx for idx, user in enumerate(user_ids)}
    idx_to_user = {idx: user for user, idx in user_to_idx.items()}

    item_to_idx = pd.Series(restaurants_df.index, index=restaurants_df['business_id']).to_dict()
    idx_to_item = {idx: item for item, idx in item_to_idx.items()}

    num_users = len(user_ids)
    num_items = len(restaurants_df)

    R = np.zeros((num_users, num_items))

    for row in train_df.itertuples():
        u_idx = user_to_idx[row.user_id]
        if row.business_id in item_to_idx:
            i_idx = item_to_idx[row.business_id]
            R[u_idx, i_idx] = row.stars

    R_df = pd.DataFrame(R)
    R_df.replace(0, np.nan, inplace=True)
    global_mean = train_df['stars'].mean()

    user_means_series = R_df.mean(axis=1, skipna=True)
    R_df = R_df.apply(lambda row: row.fillna(user_means_series[row.name]), axis=1)
    item_means = R_df.mean(skipna=True).fillna(global_mean)
    R_df = R_df.apply(lambda col: col.fillna(item_means[col.name]))
    R_df.fillna(global_mean, inplace=True)

    R_im = R_df.values

    U, s, Vt = svds(R_im, k=k)
    idx_svd = np.argsort(s)[::-1]
    U, s, Vt = U[:, idx_svd], s[idx_svd], Vt[idx_svd, :]
    predicted_ratings_cf = U @ np.diag(s) @ Vt

    return predicted_ratings_cf, R, user_to_idx, idx_to_user, item_to_idx, idx_to_item, num_items

predicted_ratings_cf, R, user_to_idx, idx_to_user, item_to_idx, idx_to_item, num_items = prepare_svd_cf(train_df, restaurants_df, k=2)

## Content-Based Filtering Setup (Features & Embeddings)

Extract our two types of features:
1. **Metadata Features:** We parse boolean attributes (like "GoodForKids"), categorical features (like "PriceRange"), and one-hot encode the restaurant categories.
2. **Semantic Text Features:** We aggregate top k reviews (based on recency) for each restaurant and pass them through a pre-trained `SentenceTransformer` (`all-mpnet-base-v2`) to embed.

Then, construct user profiles by weighted averaging the feature vectors and text embeddings of the items that a user has positively interacted with.

In [5]:
def extract_item_features(restaurants_df, train_df):
    """
    Parses restaurant attributes, categories, and review text to build normalized
    feature matrices representing the content of each item.

    Inputs:
        restaurants_df (pd.DataFrame): DataFrame containing restaurant metadata.
        train_df (pd.DataFrame): Training data containing review text.

    Return Vals:
        Tuple containing:
            - feature_matrix_normalized (np.ndarray): Normalized L2 matrix of metadata features.
            - text_matrix_normalized (np.ndarray): Normalized L2 matrix of textual semantic embeddings.
    """

    BOOLEAN_ATTRIBUTES = {'RestaurantsTakeOut': 'TakeOut', 'OutdoorSeating': 'Outdoor', 'RestaurantsDelivery': 'Deliv', 'GoodForKids': 'GFK'}
    CATEGORICAL_ATTRIBUTES = {'RestaurantsPriceRange2': 'Price', 'Ambience': 'Amb'}
    ALL_ATTRIBUTES = {**BOOLEAN_ATTRIBUTES, **CATEGORICAL_ATTRIBUTES}

    def parse_attributes(attr_str):
        if pd.isna(attr_str): return {}
        return ast.literal_eval(attr_str)

    def get_attr(row, attr_name):
        attrs = row.get('parsed_attributes', {})
        if not isinstance(attrs, dict): return 'Unknown'
        return str(attrs.get(attr_name, 'Unknown')).replace("u'", "").replace("'", "")

    restaurants_df['parsed_attributes'] = restaurants_df['attributes'].apply(parse_attributes)

    for attributes, clean_name in ALL_ATTRIBUTES.items():
        restaurants_df[clean_name] = restaurants_df.apply(lambda row: get_attr(row, attributes), axis=1)

    binary_feature_cols = []
    for col in list(BOOLEAN_ATTRIBUTES.values()):
        bin_name = col + '_Bin'
        restaurants_df[bin_name] = restaurants_df[col].map({'True': 1, 'False': 0, 'Unknown': 0, 'None': 0}).fillna(0).astype(int)
        binary_feature_cols.append(bin_name)

    categorical_clean_names = list(CATEGORICAL_ATTRIBUTES.values())
    attr_dummies = pd.get_dummies(restaurants_df[categorical_clean_names], dtype=int) if categorical_clean_names else pd.DataFrame(index=restaurants_df.index)

    restaurants_df['cat_list'] = restaurants_df['categories'].astype(str).apply(lambda x: [c.strip().lower() for c in x.split(',')])
    categories_dummies = pd.get_dummies(restaurants_df['cat_list'].explode(), dtype=int).groupby(level=0).sum()

    feature_df = pd.concat([
        (restaurants_df[binary_feature_cols] * 0.30),
        (attr_dummies * 0.50),
        (categories_dummies * 0.20)
    ], axis=1)
    feature_matrix_normalized = normalize(feature_df.values, norm='l2', axis=1)

    K_REVIEWS = 15
    MIN_WORDS = 5

    quality_reviews = train_df.dropna(subset=['text']).copy()
    quality_reviews['datetime'] = pd.to_datetime(quality_reviews['datetime'])
    quality_reviews['word_count'] = quality_reviews['text'].str.split().str.len()
    quality_reviews = quality_reviews[quality_reviews['word_count'] >= MIN_WORDS]
    quality_reviews = quality_reviews.sort_values(['business_id', 'datetime'], ascending=[True, False])

    top_k_grouped = quality_reviews.groupby('business_id').head(K_REVIEWS)
    reviews_final = top_k_grouped.groupby('business_id')['text'].apply(lambda x: ' '.join(x.astype(str))).reset_index()
    reviews_final.rename(columns={'text': 'concat_review'}, inplace=True)

    restaurants_df = restaurants_df.merge(reviews_final, on='business_id', how='left')
    restaurants_df['concat_review'] = restaurants_df['concat_review'].fillna("")

    texts_to_encode = restaurants_df['concat_review'].apply(lambda x: ' '.join(x.split()[:380])).tolist()
    model = SentenceTransformer('all-mpnet-base-v2')
    text_embeddings = model.encode(texts_to_encode, show_progress_bar=True)
    text_matrix_normalized = normalize(text_embeddings, norm='l2', axis=1)

    return feature_matrix_normalized, text_matrix_normalized

def build_user_profiles(train_df, item_to_idx, feature_matrix_normalized, text_matrix_normalized):
    """
    Constructs user profiles by averaging the features of the items they have rated.

    Inputs:
        train_df (pd.DataFrame): Training interaction data.
        item_to_idx (dict): Mapping from business_id to index.
        feature_matrix_normalized (np.ndarray): Matrix of item metadata features.
        text_matrix_normalized (np.ndarray): Matrix of item text embeddings.

    Return Vals:
        tuple: Dictionaries mapping user_id to their average metadata and text profile arrays.
    """
    all_train = train_df[train_df['stars'] >= 0.0]
    user_meta_profiles = {}
    user_text_profiles = {}

    for user, group in all_train.groupby('user_id'):
        liked_item_ids = group['business_id'].tolist()
        liked_indices = [item_to_idx[biz] for biz in liked_item_ids if biz in item_to_idx]

        if liked_indices:
            user_meta_profiles[user] = np.asarray(feature_matrix_normalized[liked_indices].mean(axis=0))
            user_text_profiles[user] = np.asarray(text_matrix_normalized[liked_indices].mean(axis=0))

    return user_meta_profiles, user_text_profiles

feature_matrix, text_matrix = extract_item_features(restaurants_df, train_df)
user_meta_profiles, user_text_profiles = build_user_profiles(train_df, item_to_idx, feature_matrix, text_matrix)

Batches: 100%|██████████| 24/24 [02:53<00:00,  7.24s/it]


## Hybrid Scoring and Prediction Generation

Now that we have both collaborative and content-based components, we calculate the hybrid score for every unrated item in the test set.

1. **CBF Score Generation:** We calculate the cosine similarity between the user's profile and the item profiles. We use a weight (`gamma`) to weight the importance of text embeddings versus categorical metadata.
2. **Scaling:** Since SVD predicted ratings and cosine sim scores live on different scales, we normalize both down to a $[0, 1]$ range using Min-Max scaling.
3. **Hybrid Combination:** We combine the two scaled scores using the `alpha` weight (similar to lecture slide example). Items already seen by the user are masked out, and the top 30 recommendations are returned.

In [6]:
def generate_hybrid_predictions(test_df, predicted_ratings_cf, user_meta_profiles, user_text_profiles, feature_matrix_normalized, text_matrix_normalized, R, user_to_idx, idx_to_item, num_items, alpha=0.65, gamma=0.25):
    """
    Combines CF and CBF scores to generate the top-N recommendations for each user in the test set.

    Inputs:
        test_df (pd.DataFrame): Testing dataset containing users to predict for.
        predicted_ratings_cf (np.ndarray): SVD collaborative filtering predictions.
        user_meta_profiles (dict): User metadata profiles.
        user_text_profiles (dict): User text semantic profiles.
        feature_matrix_normalized (np.ndarray): Item metadata profiles.
        text_matrix_normalized (np.ndarray): Item text semantic profiles.
        R (np.ndarray): Original interaction matrix for masking seen items.
        user_to_idx (dict): Mapping from user_id to row index.
        idx_to_item (dict): Mapping from column index to business_id.
        num_items (int): Total number of items.
        alpha (float, optional): Weight given to CF scores (1-alpha goes to CBF). We set it to default to 0.65.
        gamma (float, optional): Weight given to text features in CBF (1-gamma goes to meta). We set it to default to 0.25.

    Return Values:
        dict: A dictionary mapping user_id strings to a list of recommended business_id strings.
    """

    test_users = test_df['user_id'].unique()
    predictions = {}

    for user in test_users:
        if user in user_to_idx:
            u_idx = user_to_idx[user]
            cf_scores = predicted_ratings_cf[u_idx, :].copy()
        else:
            cf_scores = np.zeros(num_items)

        if user in user_meta_profiles and user in user_text_profiles:
            sim_meta = cosine_similarity(user_meta_profiles[user].reshape(1, -1), feature_matrix_normalized).flatten()
            sim_text = cosine_similarity(user_text_profiles[user].reshape(1, -1), text_matrix_normalized).flatten()
            cb_scores = (gamma * sim_text) + ((1 - gamma) * sim_meta)
        else:
            cb_scores = np.zeros(num_items)

        if cf_scores.max() != cf_scores.min():
            cf_scaled = minmax_scale(cf_scores)
        else:
            cf_scaled = cf_scores

        if cb_scores.max() != cb_scores.min():
            cb_scaled = minmax_scale(cb_scores)
        else:
            cb_scaled = cb_scores

        hybrid_scores = (alpha * cf_scaled) + ((1 - alpha) * cb_scaled)

        if user in user_to_idx:
            already_rated_indices = np.where(R[u_idx, :] > 0)[0]
            hybrid_scores[already_rated_indices] = -999.0

        if sum(hybrid_scores) > -999.0 * len(hybrid_scores):
            top_indices = hybrid_scores.argsort()[-30:][::-1]
            predictions[user] = [idx_to_item[i] for i in top_indices]
        else:
            predictions[user] = []

    return predictions

alpha = 0.65
gamma = 0.25
predictions = generate_hybrid_predictions(test_df, predicted_ratings_cf, user_meta_profiles, user_text_profiles, feature_matrix, text_matrix, R, user_to_idx, idx_to_item, num_items, alpha=alpha, gamma=gamma)

## Evaluation

Finally, we evaluate our Hybrid Recommendation System using ranking metrics (Hit@K, NDCG@K) to see how effectively our combined CF and CBF model recommends relevant items to the user.

In [7]:
metrics = evaluate_model(predictions, test_df)

results_df = pd.DataFrame([metrics]).round(4)
results_df.index = [f'Hybrid Rec (CF={alpha}, CBF={round(1-alpha,2)})']
display(results_df)

,Hit@10,Hit@20,Hit@30,NDCG@10,NDCG@20,NDCG@30
"Hybrid Rec (CF=0.65, CBF=0.35)",0.0758,0.1216,0.1666,0.0408,0.0522,0.0618
